# __НАЦИОНАЛЬНЫЙ ИССЛЕДОВАТЕЛЬСКИЙ УНИВЕРСИТЕТ ИТМО__
## Факультет программной инженерии и компьютерных технологий
### _нейрофизиология_
### __Лабораторная работа №2__
---
__выполнили:__ Егорова Варвара, Набокова Алиса, Осинкина Анастасия,
Хижниченко Мария, Шилова Ярослава, Шнейдерис Герардас

__преподаватель:__ Билый Андрей Михайлович



г. Санкт-Петербург

2025

<!-- полный [отчет](https://docs.google.com/document/d/1wLXCgsbBKiyWbK72-NTHZ4k4n9pDwDCygDSgna1js68/edit?usp=sharing) -->

### подготовка окружения и загрузка данных

In [ ]:
!pip install -q gdown # для модной загрузки данных
!pip install heartpy # по дефолту не установлен
!pip install biosppy # по дефолту не установлен
!pip install peakutils # по дефолту не установлен

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.5/159.5 kB 8.8 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# нейро штуки
import heartpy as hp
from scipy import signal
from scipy.signal import butter, filtfilt, welch
from biosppy.signals import ecg

# для загрузки файлов
from pathlib import Path
import gdown

# для модных таблиц
import re
from IPython.display import display, HTML
import numbers

FOLDER_URL = "https://drive.google.com/drive/folders/1dRvk8mXgCwOUdWDJ-sIMj2qKndefPsVX"
DATA_DIR = Path("/content/data_cached")

# поддержка кеша аче)
if DATA_DIR.exists() and any(DATA_DIR.glob("*.csv")):
    print("кэш найден — используем", DATA_DIR)
else:
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    print("скачиваем папку с Drive...")
    gdown.download_folder(FOLDER_URL, output=str(DATA_DIR), quiet=False, use_cookies=False)
    print("готово — файлы в", DATA_DIR)

csv_paths = sorted(DATA_DIR.rglob("*.csv"))

print("найдены вот столько CSV файлов:", len(csv_paths))
dfs = {
    p.stem: pd.read_csv(
        p,
        sep='\t',
        low_memory=False
    )
    for p in csv_paths
}# загрузка

#предобработка
bad_columns = ['ECG', 'EEGFR', 'EEGFL', 'EEGOR', 'EEGOL', 'PPG pulse']

for name, df in dfs.items():
    dfs[name] = df.drop(columns=['Unnamed: 10'], errors='ignore')

    for col in bad_columns:
        if col in dfs[name].columns:
            if dfs[name][col].dtype == 'object':
                dfs[name][col] = pd.to_numeric(dfs[name][col].str.replace(',', '.'), errors='coerce')
            else:
                dfs[name][col] = pd.to_numeric(dfs[name][col], errors='coerce')

    df.loc[df['ECG'].isna(), df.columns.difference(['Elapsed Time'])] = \
        df[df.columns.difference(['Elapsed Time'])].apply(pd.to_numeric, errors='coerce')\
          .rolling(5, min_periods=1).mean().shift(1).loc[df['ECG'].isna()]

    dfs[name]['ECG'] *= -1 #госпереворот
    print(name, dfs[name].shape)

скачиваем папку с Drive...


Retrieving folder contents


Processing file 1NTmMIMUe9OCCw70IL697HZFeKmsfQIfp alice.csv
Processing file 1oe25PIAT0egBD1UgBaC8HS_QauRIgiWb gera.csv
Processing file 1Q14RguOr17QmbC9KwEZxIIWkfZmXmbjo masha.csv
Processing file 1fXei0NM_KCv0fkCYPF5Qma4XlJ2bw-hN nastya.csv
Processing file 1ys8_HHvgMwzj0NwzluYgRf0o_QEOab45 varya.csv
Processing file 1rvoBJIGeZBvRPn8bLvfCZpsDzt2U0ss6 yara.csv


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1NTmMIMUe9OCCw70IL697HZFeKmsfQIfp
To: /content/data_cached/alice.csv
100%|██████████| 13.9M/13.9M [00:00<00:00, 18.3MB/s]
Downloading...
From: https://drive.google.com/uc?id=1oe25PIAT0egBD1UgBaC8HS_QauRIgiWb
To: /content/data_cached/gera.csv
100%|██████████| 13.2M/13.2M [00:00<00:00, 64.7MB/s]
Downloading...
From: https://drive.google.com/uc?id=1Q14RguOr17QmbC9KwEZxIIWkfZmXmbjo
To: /content/data_cached/masha.csv
100%|██████████| 19.0M/19.0M [00:00<00:00, 77.3MB/s]
Downloading...
From: https://drive.google.com/uc?id=1fXei0NM_KCv0fkCYPF5Qma4XlJ2bw-hN
To: /content/data_cached/nastya.csv
100%|██████████| 18.9M/18.9M [00:00<00:00, 89.0MB/s]
Downloading...
From: https://drive.google.com/uc?id=1ys8_HHvgMwzj0NwzluYgRf0o_QEOab45
To: /content/data_cached/varya.csv
100%|██████████| 17.9M/17.9M [00:01<00:00, 13.1MB/s]
Downloading...
From: http

готово — файлы в /content/data_cached
найдены вот столько CSV файлов: 6
alice (143996, 10)
gera (126927, 10)
masha (138312, 10)
nastya (137622, 10)
varya (142936, 10)
yara (130742, 10)


In [ ]:
def decor(title: str, width: int) -> str:
    return f'{width * "="} {title} {width * "="}\n' # для красоты

def _is_number(x):
    return isinstance(x, numbers.Number) # надо
dfs['gera']

,Elapsed Time,EEGFR,EEGFL,EEGOL,EEGOR,ECG,Heart Rate pulse,PPG pulse,SpO2 pulse,BioRadio Event
0,00:00:04.832,-0.101461,-0.092599,-0.091544,-0.078090,0.021727,84,81.690697,97.0,1.0
1,00:02:36.432,-0.107736,-0.091291,-0.095570,-0.079492,0.020239,74,0.000000,96.0,1.0
2,00:04:19.584,-0.114020,-0.093868,-0.101480,-0.084041,0.031912,74,0.000000,99.0,1.0
3,00:03:15.004,-0.112248,-0.093687,-0.098004,-0.082327,0.039299,120,0.000000,96.0,1.0
4,00:03:30.004,-0.114351,-0.094352,-0.099553,-0.083476,0.040398,74,0.000000,96.0,1.0
...,...,...,...,...,...,...,...,...,...,...
126922,00:08:27.688,-0.122469,-0.094588,-0.105540,-0.089116,0.007862,72,0.000000,99.0,0.0
126923,00:08:27.692,-0.122031,-0.095095,-0.105964,-0.089586,0.008354,72,0.000000,99.0,0.0
126924,00:08:27.696,-0.122158,-0.095266,-0.106149,-0.089778,0.008549,72,0.000000,99.0,0.0
126925,00:08:27.700,-0.122678,-0.094895,-0.105874,-0.089461,0.008217,72,0.000000,99.0,0.0


#### разделение данных на отрезки



In [ ]:
# индекс 2-го маркерa, по которому будут делиться данные
# 1-ый у всех маркер с индексом 1
mark_map = {
    'alice': 4,
    'masha': 5,
    'nastya': 5,
    'varya': 5,
    'yara': 4,
    'gera': 4
}
data = {}

for name, df in dfs.items():
    marks = df[df['BioRadio Event'] == 1]
    data[f"{name}_1"] = df.iloc[:marks.index[1]]
    data[f"{name}_2"] = df.iloc[marks.index[1]:marks.index[mark_map[name]]]
    data[f"{name}_3"] = df.iloc[marks.index[mark_map[name]]:]

### индексы Руфье и Руфье диксона

In [ ]:
import os

def interpret_dickson_index(index):
    if 0 <= index <= 5:
        return "Хорошая работоспособность"
    elif 5 < index <= 10:
        return "Средняя работоспособность"
    elif 10 < index <= 15:
        return "Слабая работоспособность"
    else:
        return "Неудовлетворительная работоспособность"

def interpret_ruffier_index(index):
    if 0 <= index <= 3:
        return "Отличная работоспособность"
    elif 3 <index <= 5:
        return "Хорошая работоспособность"
    elif 5 < index <= 10:
        return "Удовлетворительная работоспособность"
    elif 10 < index <= 15:
        return "Плохая работоспособность"
    else:
        return "Неудовлетворительная работоспособность"

def calculate_ruffier_from_csv(file_path):
    df = pd.read_csv(file_path, sep='\t')
    df.columns = df.columns.str.strip()
    event_indices = df[df['BioRadio Event'] == 1].index.tolist()

    p1_start, p1_end = event_indices[0], event_indices[1]

    p2_start, p2_end = event_indices[2], event_indices[3]

    p3_start, p3_end = event_indices[4], event_indices[5]

    pulse_data = pd.to_numeric(df['Heart Rate pulse'], errors='coerce').dropna()

    p1_series = pulse_data.loc[p1_start:p1_end]
    p2_series = pulse_data.loc[p2_start:p2_end]
    p3_series = pulse_data.loc[p3_start:p3_end]

    P1 = p1_series.mean()
    P2 = p2_series.mean()
    P3 = p3_series.mean()

    ruffier_index = (4*(P1 + P2 + P3) - 200) / 100
    interpretation = interpret_ruffier_index(ruffier_index)

    dickson_index = ((P2 - 70) + (P3 - P1)) / 10
    dickson_interpretation = interpret_dickson_index(dickson_index)

    return {
          "file": os.path.basename(file_path),
          "P1_rest": P1,
          "P2_after_load": P2,
          "P3_recovery": P3,
          "ruffier_index": ruffier_index,
          "interpretation": interpretation,
          "dickson_index": dickson_index,
          "dickson_interpretation": dickson_interpretation
      }

In [ ]:
import glob

csv_files = glob.glob("data_cached/*.csv")
all_results = []
for file in csv_files:
  result = calculate_ruffier_from_csv(file)
  if result:
    all_results.append(result)
    print("-" * 50)
    print(f"Результаты для файла: {result['file']}")
    print(f"  P1 (покой): {result['P1_rest']:.2f} уд/мин")
    print(f"  P2 (сразу после нагрузки): {result['P2_after_load']:.2f} уд/мин")
    print(f"  P3 (восстановление): {result['P3_recovery']:.2f} уд/мин")
    print("\n  --- Проба Руфье ---")
    print(f"  Индекс Руфье: {result['ruffier_index']:.2f}")
    print(f"  Интерпретация: {result['interpretation']}")
    print("\n  --- Проба Руфье-Диксона ---")
    print(f"  Индекс Руфье-Диксона: {result['dickson_index']:.2f}")
    print(f"  Интерпретация (Диксон): {result['dickson_interpretation']}")
    print("-" * 50 + "\n")

/tmp/ipython-input-3729225419.py:26: DtypeWarning: Columns (1,2,3,4,5,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path, sep='\t')


--------------------------------------------------
Результаты для файла: alice.csv
  P1 (покой): 92.45 уд/мин
  P2 (сразу после нагрузки): 128.18 уд/мин
  P3 (восстановление): 100.66 уд/мин

  --- Проба Руфье ---
  Индекс Руфье: 10.85
  Интерпретация: Плохая работоспособность

  --- Проба Руфье-Диксона ---
  Индекс Руфье-Диксона: 6.64
  Интерпретация (Диксон): Средняя работоспособность
--------------------------------------------------

--------------------------------------------------
Результаты для файла: gera.csv
  P1 (покой): 79.00 уд/мин
  P2 (сразу после нагрузки): 97.00 уд/мин
  P3 (восстановление): 84.50 уд/мин

  --- Проба Руфье ---
  Индекс Руфье: 8.42
  Интерпретация: Удовлетворительная работоспособность

  --- Проба Руфье-Диксона ---
  Индекс Руфье-Диксона: 3.25
  Интерпретация (Диксон): Хорошая работоспособность
--------------------------------------------------

--------------------------------------------------
Результаты для файла: nastya.csv
  P1 (покой): 83.82 уд/мин

### физиологические показатели (ЛР 1)

#### функции для вывода таблиц

In [ ]:
# группирует данные с разных участков по респондентам
def group_flat_data_simple(data_flat):
    groups = {}
    for k, v in data_flat.items():
        m = re.compile(r'^(?P<base>.+?)_([123])$').match(k)
        groups.setdefault(m.group('base'), [None, None, None])[int(m.group(2)) - 1] = v
    return groups

# собирает из данных таблицу для респондента
def build_table_for_respondent(dfs3, analyzer):
    cols = pd.MultiIndex.from_product([['участок 1', 'участок 2', 'участок 3'],
        ['max', 'min', 'mean', 'std']])
    row = []
    for df in dfs3:
        row.extend(analyzer(df))
    return pd.DataFrame([row], columns=cols)

# в ээг анализе больше данных и другой формат
def build_eeg_table_for_respondent(dfs3, analyzer):
    parts = ['участок 1', 'участок 2', 'участок 3']
    results = [{ch: (p_a, p_b, r) for ch, p_a, p_b, r in analyzer(df)} for df in dfs3]
    channels = list(results[0].keys()) if results and results[0] else sorted({c for d in results for c in d})
    table = pd.DataFrame(index=channels, columns=pd.MultiIndex.from_product([parts, ['p_α', 'p_β', 'α/β']]), dtype=float)
    for i, part in enumerate(parts):
        for ch in channels:
            vals = results[i].get(ch)
            table.loc[ch, (part, 'p_α')] = vals[0] if vals else np.nan
            table.loc[ch, (part, 'p_β')] = vals[1] if vals else np.nan
            table.loc[ch, (part, 'α/β')] = vals[2] if vals else np.nan
    return table

# собирает общую таблицу
def make_final_table(data_flat, analyzer, channels=False):
    groups = group_flat_data_simple(data_flat)
    per_respondent, names = [], []
    for name, dfs3 in groups.items():
        per_respondent.append(
            build_eeg_table_for_respondent(dfs3, analyzer) if channels
            else build_table_for_respondent(dfs3, analyzer)
            )
        names.append(name)
    return pd.concat(per_respondent, keys=names, names=['респондент', 'канал'])

# ------------------------ стили и отображение ------------------------
TABLE_STYLES = [
    {'selector': 'td', 'props': [('padding', '6px'), ('vertical-align', 'middle'), ('border-right', '1px solid #666')]},
    {'selector': 'th.col_heading.level1', 'props': [('padding', '6px'), ('text-align', 'center'), ('border-right', '1px solid #666')]},
    {'selector': 'th.col_heading.level0', 'props': [('padding', '8px'), ('text-align', 'center'), ('font-weight', '600'), ('border-right', '3px solid #444')]},
]

def _channel_bg_style(channel_name):
    return f"background-color: hsl({abs(hash(channel_name)) % 360}deg 50% 88%); color: #000;"

def color_row_by_channel(row):
    return [_channel_bg_style(row.name[1] if isinstance(row.name, tuple) and len(row.name) > 1 else str(row.name)) for _ in row]

def _apply_styles(styler, float_fmt="{:.5e}"):
    return styler.format(lambda v: float_fmt.format(v) if pd.notna(v) else "").set_table_styles(TABLE_STYLES)

def show(final_df, float_fmt="{:.5e}"):
    display(_apply_styles(final_df.droplevel(level=1).style, float_fmt))

def show_eeg(final_df, float_fmt="{:.5e}"):
    display(_apply_styles(final_df.style.apply(color_row_by_channel, axis=1), float_fmt))

#### функции для расчета показателей

In [ ]:
fs = 250
eeg_cols = ['EEGFR', 'EEGFL', 'EEGOR', 'EEGOL']

def band_power(signal, fs, band, nperseg=1024):
    # Вычисление мощности в заданной полосе частот через методом Уэлча
    f, Pxx = welch(signal, fs=fs, nperseg=min(nperseg, len(signal)))
    idx = (f >= band[0]) & (f < band[1])
    return np.trapezoid(Pxx[idx], f[idx]) if np.any(idx) else 0.0

alpha_band = (8, 13)
beta_band = (13, 30)

def eeg_rhythms_analyze(df: pd.DataFrame) -> list:
    return [
        (ch, P_alpha, P_beta, np.nan if P_beta == 0 else P_alpha / P_beta)
        for ch in list(filter(lambda ch: ch in df.columns, eeg_cols))
        for sig in [pd.to_numeric(df[ch], errors='coerce').dropna().to_numpy()]
        if len(sig) > 0
        for P_alpha in [band_power(sig, fs, alpha_band)]
        for P_beta in [band_power(sig, fs, beta_band)]
    ]

def series_stats(row: pd.Series) -> tuple:
    num_row = pd.to_numeric(row, errors='coerce').dropna()
    if num_row.empty:
        return np.nan, np.nan, np.nan, np.nan
    return num_row.max(), num_row.min(), num_row.mean(), num_row.std(ddof=0)

def bpm_analyze(df: pd.DataFrame) -> tuple:
    return series_stats(df['Heart Rate pulse'])

def spo2_analyze(df: pd.DataFrame) -> tuple:
    return series_stats(df['SpO2 pulse'])

# ВСР

def bandpass_filter(signal_data, lowcut, highcut, fs: int):
    nyq = 0.5 * fs
    b, a = signal.butter(2, [lowcut / nyq, highcut / nyq], btype='band')
    return signal.filtfilt(b, a, signal_data)

def compute_baevsky_index(rr_intervals):
    hist, bin_edges = np.histogram(np.sort(rr_intervals), bins=50)
    Mo_index = np.argmax(hist)
    Mo = (bin_edges[Mo_index] + bin_edges[Mo_index + 1]) / 2
    AMo = hist[Mo_index] / len(rr_intervals) * 100
    MxDMn = np.max(rr_intervals) - np.min(rr_intervals)

    if Mo * MxDMn == 0: return np.nan
    return AMo / (2 * Mo * MxDMn) * 1e6  # масштабируем

def compute_frequency_domain(rr_intervals, fs=4.0) -> dict:
    # интерполяция RR для равномерной сетки
    time = np.cumsum(rr_intervals) / 1000.0
    time -= time[0] # сдвиг к нулю
    f_interp = np.interp(np.arange(0, time[-1], 1/fs), time, rr_intervals)
    f_interp -= np.mean(f_interp)

    freqs, psd = signal.welch(f_interp, fs=fs, nperseg=len(f_interp)//2)

    # диапазоны
    def _band_power(fmin, fmax):
        mask = (freqs >= fmin) & (freqs < fmax)
        return np.trapezoid(psd[mask], freqs[mask])

    vlf = _band_power(0.003, 0.04)
    lf = _band_power(0.04, 0.15)
    hf = _band_power(0.15, 0.4)
    ulf = _band_power(0.0, 0.003)
    total_power = vlf + lf + hf + ulf
    lf_hf = lf / hf if hf > 0 else np.nan

    return {'ULF': ulf, 'VLF': vlf, 'LF': lf, 'HF': hf, 'LF/HF': lf_hf, 'Total': total_power}

def ecg_analyze(df: pd.DataFrame):
    ecg_data = df['ECG'].astype(float).values
    ecg_data = ecg_data[np.isfinite(ecg_data)]
    ecg_data = bandpass_filter(ecg_data, 0.5, 40, 250)

    out = ecg.ecg(signal=ecg_data, sampling_rate=250, show=False)
    rr_intervals = np.diff(out['rpeaks']) / 250 * 1000  # RR в мс

    # временные показатели
    sdnn = np.std(rr_intervals)
    rmssd = np.sqrt(np.mean(np.diff(rr_intervals)**2))

    # частотные показатели
    freq = compute_frequency_domain(rr_intervals)
    si = compute_baevsky_index(rr_intervals)

    return freq, si

#### Альфа и бета ритмы

In [ ]:
print(decor('анализ данных по альфа и бета ритмам ЭЭГ', 35))
show_eeg(make_final_table(data, eeg_rhythms_analyze, channels=True), float_fmt="{:.3e}")

=================================== анализ данных по альфа и бета ритмам ЭЭГ ===================================



#### ЧСС

In [ ]:
print(decor('анализ данных о ЧСС', 35))
show(make_final_table(data, bpm_analyze), float_fmt="{:.2f}")

=================================== анализ данных о ЧСС ===================================



#### оксигенация

In [ ]:
print(decor('анализ данных об уровне оксигенации', 24))
show(make_final_table(data, spo2_analyze), float_fmt="{:.2f}")

======================== анализ данных об уровне оксигенации ========================



#### ВСР

In [ ]:
#ECG
print(decor('анализ показателей вариативности сердечного ритма', 35))
for name, df in data.items():
    print(f"\n{name}")

    freq, si = ecg_analyze(df)

    print(f"ULF: {freq['ULF']:.2f}, VLF: {freq['VLF']:.2f}, LF: {freq['LF']:.2f}, HF: {freq['HF']:.2f}, LF/HF: {freq['LF/HF']:.2f}")
    print(f"Total Power (VSR): {freq['Total']:.2f}")
    print(f"Индекс напряжения по Баевскому (SI): {si:.2f}")

=================================== анализ показателей вариативности сердечного ритма ===================================


alice_1
ULF: 0.00, VLF: 123017.91, LF: 77632.04, HF: 50288.37, LF/HF: 1.54
Total Power (VSR): 250938.32
Индекс напряжения по Баевскому (SI): 6.57

alice_2
ULF: 0.00, VLF: 8718.27, LF: 12583.95, HF: 20627.13, LF/HF: 0.61
Total Power (VSR): 41929.35
Индекс напряжения по Баевскому (SI): 14.78

alice_3
ULF: 0.00, VLF: 905.61, LF: 619.62, HF: 305.70, LF/HF: 2.03
Total Power (VSR): 1830.93
Индекс напряжения по Баевскому (SI): 18.43

gera_1
ULF: 0.00, VLF: 1651.70, LF: 2387.05, HF: 752.34, LF/HF: 3.17
Total Power (VSR): 4791.09
Индекс напряжения по Баевскому (SI): 8.95

gera_2
ULF: 0.00, VLF: 29605.22, LF: 23498.81, HF: 36205.72, LF/HF: 0.65
Total Power (VSR): 89309.75
Индекс напряжения по Баевскому (SI): 5.71

gera_3
ULF: 0.00, VLF: 663.71, LF: 3265.91, HF: 745.34, LF/HF: 4.38
Total Power (VSR): 4674.97
Индекс напряжения по Баевскому (SI): 7.23

masha_1
ULF: 0.00, VLF: 

In [ ]:
from IPython.display import HTML, display
import pandas as pd

# Заголовок
print("Анализ показателей вариативности сердечного ритма".center(60))

# Собираем результаты
results = []
for name, df in data.items():
    freq, si = ecg_analyze(df)
    results.append({
        'Пациент / Сегмент': name,
        'ULF': f"{freq['ULF']:.2f}",
        'VLF': f"{freq['VLF']:.2f}",
        'LF':  f"{freq['LF']:.2f}",
        'HF':  f"{freq['HF']:.2f}",
        'LF/HF': f"{freq['LF/HF']:.2f}",
        'Total Power (VSR)': f"{freq['Total']:.2f}",
        'SI (Баевский)': f"{si:.2f}"
    })

# DataFrame → HTML
df_results = pd.DataFrame(results)

# Простая таблица: минимум CSS
html = """
<style>
  .simple-table {
    width: 100%;
    max-width: 900px;
    margin: 20px auto;
    border-collapse: collapse;
    font-family: Arial, sans-serif;
    font-size: 14px;
  }
  .simple-table th {
    background-color: #f0f0f0;
    padding: 10px;
    text-align: center;
    border: 1px solid #ddd;
  }
  .simple-table td {
    padding: 8px 10px;
    text-align: center;
    border: 1px solid #ddd;
  }
  .simple-table tr:nth-child(even) {
    background-color: #f9f9f9;
  }
</style>

<table class="simple-table">
""" + df_results.to_html(index=False, border=0) + "</table>"

display(HTML(html))

     Анализ показателей вариативности сердечного ритма      


Пациент / Сегмент,ULF,VLF,LF,HF,LF/HF,Total Power (VSR),SI (Баевский)
alice_1,0.00,123017.91,77632.04,50288.37,1.54,250938.32,6.57
alice_2,0.00,8718.27,12583.95,20627.13,0.61,41929.35,14.78
alice_3,0.00,905.61,619.62,305.70,2.03,1830.93,18.43
gera_1,0.00,1651.70,2387.05,752.34,3.17,4791.09,8.95
gera_2,0.00,29605.22,23498.81,36205.72,0.65,89309.75,5.71
gera_3,0.00,663.71,3265.91,745.34,4.38,4674.97,7.23
masha_1,0.00,1207.23,903.04,340.39,2.65,2450.65,10.93
masha_2,0.00,2312.51,6070.91,14527.96,0.42,22911.38,8.61
masha_3,0.00,540.51,433.19,68.70,6.31,1042.41,24.29
nastya_1,0.00,267.10,1171.50,1202.05,0.97,2640.65,9.47


In [ ]:
print(decor('анализ показателей вариативности сердечного ритма по ФПГ', 35))
for name, df in data.items():
    print(f"\n{name}")
    ppg_data = df['PPG pulse'].astype(str).str.replace(',', '.').astype(float).values
    ppg_data = ppg_data[np.isfinite(ppg_data)]
    ppg_data = bandpass_filter(ppg_data, 0.5, 8, 100)  # PPG: ниже частоты, fs=100Hz

    try:
        # Детекция пиков PPG (аналог R-пиков в ЭКГ)
        peaks, _ = signal.find_peaks(ppg_data, distance=0.5*100, prominence=np.std(ppg_data)*0.3)
        rr_intervals = np.diff(peaks) / 100 * 1000  # PP-интервалы в мс

        if len(rr_intervals) < 5:
            raise ValueError("мало пиков для анализа")

        # Частотные показатели
        freq = compute_frequency_domain(rr_intervals)
        si = compute_baevsky_index(rr_intervals)

        print(f"ULF: {freq['ULF']:.2f}, VLF: {freq['VLF']:.2f}, LF: {freq['LF']:.2f}, HF: {freq['HF']:.2f}, LF/HF: {freq['LF/HF']:.2f}")
        print(f"Total Power (VSR): {freq['Total']:.2f}")
        print(f"Индекс напряжения по Баевскому (SI): {si:.2f}")

    except Exception as e:
        print(f"ошибка у {name}: {e}")

=================================== анализ показателей вариативности сердечного ритма по ФПГ ===================================


alice_1
ULF: 0.00, VLF: 4814090.56, LF: 4731193.62, HF: 623708.13, LF/HF: 7.59
Total Power (VSR): 10168992.31
Индекс напряжения по Баевскому (SI): 1.07

alice_2
ошибка у alice_2: мало пиков для анализа

alice_3
ошибка у alice_3: мало пиков для анализа

gera_1
ULF: 0.00, VLF: 1929.55, LF: 3533.80, HF: 3125.61, LF/HF: 1.13
Total Power (VSR): 8588.95
Индекс напряжения по Баевскому (SI): 13.09

gera_2
ошибка у gera_2: мало пиков для анализа

gera_3
ошибка у gera_3: мало пиков для анализа

masha_1
ULF: 0.00, VLF: 4715.56, LF: 3923.13, HF: 927.38, LF/HF: 4.23
Total Power (VSR): 9566.07
Индекс напряжения по Баевскому (SI): 9.76

masha_2
ULF: 0.00, VLF: 4512.81, LF: 19457.75, HF: 19100.90, LF/HF: 1.02
Total Power (VSR): 43071.45
Индекс напряжения по Баевскому (SI): 2.43

masha_3
ULF: 475.96, VLF: 24959.90, LF: 22523.28, HF: 30923.14, LF/HF: 0.73
Total Power (VSR): 

In [ ]:
from IPython.display import HTML, display
import pandas as pd
import numpy as np
from scipy import signal

def decor(text, length=35):
    return f"{text}".center(length, " ")

print(decor('анализ показателей вариативности сердечного ритма по ФПГ', 60))

results = []

for name, df in data.items():
    try:
        ppg_data = df['PPG pulse'].astype(str).str.replace(',', '.').astype(float).values
        ppg_data = ppg_data[np.isfinite(ppg_data)]
        ppg_data = bandpass_filter(ppg_data, 0.5, 8, 100)  # fs=100 Hz

        peaks, _ = signal.find_peaks(ppg_data, distance=0.5*100, prominence=np.std(ppg_data)*0.3)
        rr_intervals = np.diff(peaks) / 100 * 1000  # в мс

        if len(rr_intervals) < 5:
            raise ValueError("мало пиков для анализа")

        freq = compute_frequency_domain(rr_intervals)
        si = compute_baevsky_index(rr_intervals)

        results.append({
            'Пациент / Сегмент': name,
            'ULF': f"{freq['ULF']:.2f}",
            'VLF': f"{freq['VLF']:.2f}",
            'LF':  f"{freq['LF']:.2f}",
            'HF':  f"{freq['HF']:.2f}",
            'LF/HF': f"{freq['LF/HF']:.2f}",
            'Total Power (VSR)': f"{freq['Total']:.2f}",
            'SI (Баевский)': f"{si:.2f}"
        })

    except Exception as e:
        results.append({
            'Пациент / Сегмент': name,
            'ULF': '—',
            'VLF': '—',
            'LF':  '—',
            'HF':  '—',
            'LF/HF': '—',
            'Total Power (VSR)': '—',
            'SI (Баевский)': f'Ошибка: {e}'
        })

df_results = pd.DataFrame(results)

html = """
<style>
  .simple-table {
    width: 100%;
    max-width: 950px;
    margin: 20px auto;
    border-collapse: collapse;
    font-family: Arial, sans-serif;
    font-size: 14px;
  }
  .simple-table th {
    background-color: #f0f0f0;
    padding: 10px;
    text-align: center;
    border: 1px solid #ddd;
    font-weight: 600;
  }
  .simple-table td {
    padding: 8px 10px;
    text-align: center;
    border: 1px solid #ddd;
  }
  .simple-table tr:nth-child(even) {
    background-color: #f9f9f9;
  }
  .error-row {
    color: #d9534f;
    font-style: italic;
  }
</style>

<table class="simple-table">
""" + df_results.to_html(index=False, border=0, escape=False) + "</table>"

html = html.replace('Ошибка:', '<span class="error-row">Ошибка:</span>')

display(HTML(html))

  анализ показателей вариативности сердечного ритма по ФПГ  


Пациент / Сегмент,ULF,VLF,LF,HF,LF/HF,Total Power (VSR),SI (Баевский)
alice_1,0.00,4814090.56,4731193.62,623708.13,7.59,10168992.31,1.07
alice_2,—,—,—,—,—,—,Ошибка: мало пиков для анализа
alice_3,—,—,—,—,—,—,Ошибка: мало пиков для анализа
gera_1,0.00,1929.55,3533.80,3125.61,1.13,8588.95,13.09
gera_2,—,—,—,—,—,—,Ошибка: мало пиков для анализа
gera_3,—,—,—,—,—,—,Ошибка: мало пиков для анализа
masha_1,0.00,4715.56,3923.13,927.38,4.23,9566.07,9.76
masha_2,0.00,4512.81,19457.75,19100.90,1.02,43071.45,2.43
masha_3,475.96,24959.90,22523.28,30923.14,0.73,78882.27,8.71
nastya_1,0.00,1656.81,3989.26,957.09,4.17,6603.16,7.59


После физической нагрузки у большинства респондентов увеличился LF/HF, что свидетельствует о доминанте симпатической активности. Восстановление шло медленно, HF-компонент оставался сниженным. Индекс Баевского повышался, что как раз указывает на напряжение в организме в связи с физической нагрузкой

## выводы по работе:
Не все физиологические данные были сняты корректно, но это не помешало нам комплексно оценить адаптацию сердечно-сосудистой и нервной системы.

1) После физической нагрузки наблюдается закономерный стрессовый сдвиг:
  - увеличение чсс
  - рост LF/HF и индекса Баевского
  - снижение α/β на ЭЭГ

2) Восстановление замдленно у большинства участников, что свидетельствует о слабой физической подготовке. Но мы наглядно увидели как воспалительные процессы в организме могут влиять на физиологические покзатели, а именно повышенный пульс в покое и медленное его восстановление после физической нагрузки